In [14]:
# hyperparameter tuning

# Import necessary libraries
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import os
import dagshub

In [15]:
mlflow.set_tracking_uri("https://dagshub.com/kush2501/mlops-mini-project.mlflow")
dagshub.init(repo_owner='kush2501', repo_name='mlops-mini-project', mlflow=True)

Initialized MLflow to track repo "kush2501/mlops-mini-project"

Repository kush2501/mlops-mini-project initialized!

In [16]:
# Load the data
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])


In [17]:
# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [18]:
# Normalize the text data
df = normalize_text(df)

In [19]:
df.head()

,sentiment,content
0,empty,tiffanylue know listenin bad habit earlier sta...
1,sadness,layin n bed headache ughhhh waitin call
2,sadness,funeral ceremony gloomy friday
3,enthusiasm,want hang friend soon
4,neutral,dannycastillo want trade someone houston ticke...


In [20]:
# keep only sadness and happiness
df = df[df['sentiment'].isin(['happiness','sadness'])]

# encode labels
df['sentiment'] = df['sentiment'].map({
    'sadness':0,
    'happiness':1
})

In [21]:
df = df[df['content'].str.strip() != ""]
df['content'] = df['content'].astype(str)

In [23]:
# 🔹 Step 1: Raw data
X = df['content']
y = df['sentiment']

# 🔹 Step 2: Split FIRST ✅
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 🔹 Step 3: Fit ONLY on train ✅
vectorizer = CountVectorizer()

X_train_vec = vectorizer.fit_transform(X_train)   # ✅ learn from train
X_test_vec = vectorizer.transform(X_test)         # ✅ apply on test

In [25]:
# Set the experiment name
mlflow.set_experiment("LogisticRegression_Hyperparameter_Tuning")

2026/03/18 14:55:46 INFO mlflow.tracking.fluent: Experiment with name 'LogisticRegression_Hyperparameter_Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/278e1116716640e992d050aeac7ceac0', creation_time=1773825946615, experiment_id='6', last_update_time=1773825946615, lifecycle_stage='active', name='LogisticRegression_Hyperparameter_Tuning', tags={}, workspace='default'>

In [26]:
# Define hyperparameter grid for Logistic Regression
param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

In [28]:
# Start the parent run for hyperparameter tuning
with mlflow.start_run():

    # Perform grid search
    grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5, scoring='f1', n_jobs=-1)
    grid_search.fit(X_train_vec, y_train)

    # Log each parameter combination as a child run
    for params, mean_score, std_score in zip(grid_search.cv_results_['params'], grid_search.cv_results_['mean_test_score'], grid_search.cv_results_['std_test_score']):
        with mlflow.start_run(run_name=f"LR with params: {params}", nested=True):
            model = LogisticRegression(**params)
            model.fit(X_train_vec, y_train)
            
            # Model evaluation
            y_pred = model.predict(X_test_vec)
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred)
            recall = recall_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            
            # Log parameters and metrics
            mlflow.log_params(params)
            mlflow.log_metric("mean_cv_score", mean_score)
            mlflow.log_metric("std_cv_score", std_score)
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)
            
         # 🔹 Step 8: Log ORIGINAL DATA (Not vectorized ❗)
            train_df = pd.DataFrame({"text": X_train.reset_index(drop=True),"target": y_train.reset_index(drop=True)})
            test_df = pd.DataFrame({"text": X_test.reset_index(drop=True),"target": y_test.reset_index(drop=True)})

            mlflow.log_input(mlflow.data.from_pandas(train_df),context="training")
            mlflow.log_input(mlflow.data.from_pandas(test_df),context="validation")


            # Print the results for verification
            print(f"Mean CV Score: {mean_score}, Std CV Score: {std_score}")
            print(f"Accuracy: {accuracy}")
            print(f"Precision: {precision}")
            print(f"Recall: {recall}")
            print(f"F1 Score: {f1}")

    # Log the best run details in the parent run
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    mlflow.log_params(best_params)
    mlflow.log_metric("best_f1_score", best_score)
    
    print(f"Best Params: {best_params}")
    print(f"Best F1 Score: {best_score}")

    # Save and log the notebook
    mlflow.log_artifact("exp3_lor_bow_hp.ipynb")

    # Log model
    mlflow.sklearn.log_model(grid_search.best_estimator_, name="model")

d:\mlops\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\mlops\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\mlops\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please us

Mean CV Score: 0.7031332131344762, Std CV Score: 0.013927795334772935
Accuracy: 0.7383132530120482
Precision: 0.7771362586605081
Recall: 0.6578690127077224
F1 Score: 0.7125463208046585
🏃 View run LR with params: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/6/runs/28a0966940684efeb8a6a9e415932d50
🧪 View experiment at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/6


d:\mlops\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\mlops\mlops-mini-project\myenv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing value

Mean CV Score: 0.7813410147858625, Std CV Score: 0.01874604674794475
Accuracy: 0.7816867469879518
Precision: 0.7844311377245509
Recall: 0.7683284457478006
F1 Score: 0.7762962962962963
🏃 View run LR with params: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/6/runs/b565c6cd1dc64d4997c91ee839ae044a
🧪 View experiment at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/6


d:\mlops\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\mlops\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
d:\mlops\mlops-mini-project\myenv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is

...

Mean CV Score: 0.7716147165411582, Std CV Score: 0.008954249971253414
Accuracy: 0.7624096385542168
Precision: 0.7562862669245648
Recall: 0.7644183773216031
F1 Score: 0.7603305785123967
🏃 View run LR with params: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/6/runs/91738e7ffdef4b64b02c36269fb48e08
🧪 View experiment at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/6


d:\mlops\mlops-mini-project\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\mlops\mlops-mini-project\myenv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing value

Mean CV Score: 0.7782818313965695, Std CV Score: 0.008437121739649023
Accuracy: 0.7763855421686747
Precision: 0.7664442326024785
Recall: 0.7859237536656891
F1 Score: 0.7760617760617761
🏃 View run LR with params: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/6/runs/e65d2779139b470dafc42bd81e07d509
🧪 View experiment at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/6
Best Params: {'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}
Best F1 Score: 0.7878830040101602


2026/03/18 14:58:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run chill-bug-192 at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/6/runs/730b0d05177249bab15697af739ae557
🧪 View experiment at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/6
